In [1]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [2]:
import os, shutil

if os.path.exists("omnisense"):
    shutil.rmtree("omnisense")

!git clone https://github.com/cksajil/omnisense.git
%cd omnisense
!git log --oneline

Cloning into 'omnisense'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 72 (delta 24), reused 57 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 269.14 KiB | 2.15 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/omnisense
2a5f21e (HEAD -> main, origin/main, origin/HEAD) added notebook
58c877f Merge pull request #2 from cksajil/feat/nlp-pipeline
7554425 (origin/feat/nlp-pipeline) fix(deps): Intel Mac compatibility fixes
db64511 Merge pull request #1 from cksajil/feat/audio-pipeline
ea4c3a6 (origin/feat/audio-pipeline) feat(audio): implement Whisper transcription pipeline
b1ddbf1 feat: initial project scaffold
5679578 Initial commit


In [3]:
%%capture
!pip install faster-whisper \
             sentence-transformers \
             faiss-cpu \
             gradio \
             python-dotenv \
             loguru \
             pytube \
             opencv-python-headless \
             ffmpeg-python \
             accelerate \
             sentencepiece

In [4]:
!pip install -e . --quiet

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for omnisense (pyproject.toml) ... done


In [5]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]  = userdata.get("HF_TOKEN")
os.environ["DEVICE"]    = "cuda"
os.environ["LOG_LEVEL"] = "INFO"

print(f"HF_TOKEN : {'✓' if os.environ.get('HF_TOKEN') else '✗ NOT SET'}")
print(f"Device   : {os.environ['DEVICE']}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")

HF_TOKEN : ✓
Device   : cuda
GPU      : Tesla T4


In [6]:
import sys, numpy as np, torch, transformers, faster_whisper, omnisense

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"FasterWhisper: {faster_whisper.__version__}")
print(f"OmniSense    : {omnisense.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print()
print("All imports OK ✓")

Python       : 3.12.12
NumPy        : 2.0.2
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
FasterWhisper: 1.2.1
OmniSense    : 0.1.0
CUDA         : True

All imports OK ✓


In [7]:
from omnisense.pipelines.audio import AudioPipeline

audio = AudioPipeline(device="cuda")
audio_result = audio("/content/sample.wav")

print(f"Language  : {audio_result['language']}")
print(f"Duration  : {audio_result['duration']:.1f}s")
print(f"Segments  : {len(audio_result['segments'])}")
print(f"Transcript: {audio_result['transcript'][:300]}")
print()
print("Audio pipeline PASSED ✓")

2026-03-23 13:24:15 | INFO     | omnisense.pipelines.base:24 — AudioPipeline initialised on device=cuda
2026-03-23 13:24:15 | INFO     | omnisense.pipelines.base:36 — Auto-loading AudioPipeline…
2026-03-23 13:24:15 | INFO     | omnisense.pipelines.audio:50 — Loading faster-whisper model 'base'…
2026-03-23 13:24:18 | INFO     | omnisense.pipelines.audio:57 — faster-whisper model loaded ✓
2026-03-23 13:24:19 | INFO     | omnisense.pipelines.audio:72 — Transcribing sample.wav (5.6s)…
2026-03-23 13:24:20 | INFO     | omnisense.pipelines.audio:99 — Transcription complete — 2 segments, 1 chunks, language=en


Language  : en
Duration  : 5.6s
Segments  : 2
Transcript: Hello this is a test of the Omniscience Audio Pipeline using faster whisper on an Intel mic

Audio pipeline PASSED ✓


In [8]:
from omnisense.pipelines.nlp import NLPPipeline

nlp = NLPPipeline(device="cuda")
nlp_result = nlp(audio_result)

print(f"Summary    : {nlp_result['summary']}")
print(f"Top topic  : {nlp_result['top_topic']}")
print(f"Word count : {nlp_result['word_count']}")
print()
print("Entities:")
for ent in nlp_result['entities'][:5]:
    print(f"  [{ent['label']}] {ent['text']} (score={ent['score']:.2f})")
print()
print("Topics:")
for t in nlp_result['topics'][:3]:
    print(f"  {t['label']:<20} {t['score']:.4f}")
print()
print("NLP pipeline PASSED ✓")

2026-03-23 13:24:32 | INFO     | omnisense.pipelines.base:24 — NLPPipeline initialised on device=cuda
2026-03-23 13:24:32 | INFO     | omnisense.pipelines.base:36 — Auto-loading NLPPipeline…
2026-03-23 13:24:32 | INFO     | omnisense.pipelines.nlp:83 — Loading NLP models — this may take a few minutes on first run…
2026-03-23 13:24:32 | INFO     | omnisense.pipelines.nlp:85 —   [1/3] Loading summarizer: sshleifer/distilbart-cnn-6-6


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-03-23 13:24:57 | INFO     | omnisense.pipelines.nlp:107 — All NLP models loaded ✓
2026-03-23 13:24:57 | INFO     | omnisense.pipelines.nlp:142 — Running NLP on 1 chunk(s)…
2026-03-23 13:24:57 | INFO     | omnisense.pipelines.nlp:149 — Summarisation complete — 1 chunk summaries
2026-03-23 13:24:57 | INFO     | omnisense.pipelines.nlp:154 — NER complete — 4 unique entities found
2026-03-23 13:24:58 | INFO     | omnisense.pipelines.nlp:158 — Classification complete — top topic: technology


Summary    : Hello this is a test of the Omniscience Audio Pipeline using faster whisper on an Intel mic
Top topic  : technology
Word count : 17

Entities:
  [MISC] O (score=0.98)
  [ORG] Intel (score=0.92)
  [MISC] Audio Pipeline (score=0.83)
  [MISC] cie (score=0.58)

Topics:
  technology           0.8538
  science              0.0415
  environment          0.0383

NLP pipeline PASSED ✓


In [11]:
from importlib import reload
import omnisense.pipelines.vision as vmod
reload(vmod)
from omnisense.pipelines.vision import VisionPipeline

vision = VisionPipeline(device="cuda")
vision_result = vision("/content/189_List_Comprehension.mov", max_frames=10)

print(f"Frames processed : {vision_result['frame_count']}")
print(f"Top visual label : {vision_result['top_visual_label']}")
print(f"Unique objects   : {vision_result['unique_objects'][:8]}")
print()
print("Sample captions:")
for cap in vision_result["captions"][:3]:
    print(f"  [{cap['timestamp']:.1f}s] {cap['caption']}")
print()
print("Vision pipeline PASSED ✓")

2026-03-23 13:27:38 | INFO     | omnisense.pipelines.base:24 — VisionPipeline initialised on device=cuda
2026-03-23 13:27:38 | INFO     | omnisense.pipelines.base:36 — Auto-loading VisionPipeline…
2026-03-23 13:27:38 | INFO     | omnisense.pipelines.vision:167 — Loading vision models…
2026-03-23 13:27:38 | INFO     | omnisense.pipelines.vision:169 —   [1/3] CLIP: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

2026-03-23 13:28:13 | INFO     | omnisense.pipelines.vision:192 — All vision models loaded ✓
2026-03-23 13:28:13 | INFO     | omnisense.utils.vision:46 — Extracting frames from 189_List_Comprehension.mov (192.6s, 59.0fps, interval=58)
Error during conversion: AttributeError("'str' object has no attribute 'decode'")
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Frames processed : 10
Top visual label : presentation or lecture
Unique objects   : ['cell phone']

Sample captions:
  [0.0s] a screenshot of a black screen with a blue arrow in the middle
  [1.0s] a screenshot of a black screen with a blue arrow in the middle
  [2.0s] a screenshot of a black screen with a blue arrow in the middle

Vision pipeline PASSED ✓
